# Module 4: Search in Azure DocumentDB (C# completed reference)

In [ ]:
#r "nuget: MongoDB.Driver, 3.4.0"using MongoDB.Bson;using MongoDB.Driver;using System.Linq;var cs=Environment.GetEnvironmentVariable("DOCUMENTDB_CONNECTION_STRING");if(string.IsNullOrWhiteSpace(cs)) throw new Exception("Set DOCUMENTDB_CONNECTION_STRING");var client=new MongoClient(cs);var db=client.GetDatabase("docdbworkshop");var collection=db.GetCollection<BsonDocument>("workshop_content");db.RunCommand<BsonDocument>(new BsonDocument("ping",1))

In [ ]:
collection.DeleteMany(FilterDefinition<BsonDocument>.Empty);collection.InsertMany(new[]{ new BsonDocument{{"_id","doc-search-001"},{"title","DiskANN vector indexing"},{"category","vector"},{"body","Azure DocumentDB supports DiskANN vector indexes for high recall semantic similarity search over embeddings stored with documents."},{"sku","SEARCH-VEC-001"},{"embedding",new BsonArray{0.92,0.80,0.18}}}, new BsonDocument{{"_id","doc-search-002"},{"title","BM25 keyword search"},{"category","full-text"},{"body","Azure DocumentDB full-text search ranks keyword matches with BM25 and exposes scores through searchScore metadata."},{"sku","SEARCH-FTS-001"},{"embedding",new BsonArray{0.20,0.12,0.94}}}, new BsonDocument{{"_id","doc-search-003"},{"title","Hybrid search with RRF"},{"category","hybrid"},{"body","Hybrid search combines BM25 keyword results with vector results and fuses the ranked lists using Reciprocal Rank Fusion."},{"sku","SEARCH-HYB-001"},{"embedding",new BsonArray{0.76,0.70,0.42}}}, new BsonDocument{{"_id","doc-search-004"},{"title","RAG grounding"},{"category","rag"},{"body","Retrieval augmented generation retrieves relevant chunks from Azure DocumentDB and grounds the model answer in that context."},{"sku","RAG-PIPE-001"},{"embedding",new BsonArray{0.82,0.74,0.36}}}});collection.CountDocuments(FilterDefinition<BsonDocument>.Empty)

In [ ]:
db.RunCommand<BsonDocument>(new BsonDocument{{"createIndexes","workshop_content"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_embedding_diskann"},{"key",new BsonDocument("embedding","cosmosSearch")},{"cosmosSearchOptions",new BsonDocument{{"kind","vector-diskann"},{"dimensions",3},{"similarity","COS"},{"maxDegree",32},{"lBuild",64}}}}}}});db.RunCommand<BsonDocument>(new BsonDocument{{"createSearchIndexes","workshop_content"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_body_fts"},{"definition",new BsonDocument("mappings",new BsonDocument{{"dynamic",false},{"fields",new BsonDocument("body",new BsonDocument("type","string"))}})}}}}});

## Vector search

In [ ]:
var v=new BsonArray{0.90,0.78,0.22};collection.Aggregate<BsonDocument>(new[]{new BsonDocument("$search",new BsonDocument("cosmosSearch",new BsonDocument{{"path","embedding"},{"vector",v},{"k",3}})),new BsonDocument("$project",new BsonDocument{{"_id",1},{"title",1},{"score",new BsonDocument("$meta","searchScore")}})}).ToList()

## BM25 keyword search

In [ ]:
collection.Aggregate<BsonDocument>(new[]{new BsonDocument("$search",new BsonDocument{{"index","idx_body_fts"},{"text",new BsonDocument{{"query","BM25 ranking"},{"path","body"}}}}),new BsonDocument("$limit",5),new BsonDocument("$project",new BsonDocument{{"_id",1},{"title",1},{"score",new BsonDocument("$meta","searchScore")}})}).ToList()

## Fuzzy search

In [ ]:
collection.Aggregate<BsonDocument>(new[]{new BsonDocument("$search",new BsonDocument{{"index","idx_body_fts"},{"text",new BsonDocument{{"query","retrival augmentd genration"},{"path","body"},{"fuzzy",new BsonDocument("maxEdits",1)}}}}),new BsonDocument("$limit",5),new BsonDocument("$project",new BsonDocument{{"_id",1},{"title",1},{"score",new BsonDocument("$meta","searchScore")}})}).ToList()

## Phrase search

In [ ]:
collection.Aggregate<BsonDocument>(new[]{new BsonDocument("$search",new BsonDocument{{"index","idx_body_fts"},{"phrase",new BsonDocument{{"query","Reciprocal Rank Fusion"},{"path","body"},{"slop",0}}}}),new BsonDocument("$limit",5),new BsonDocument("$project",new BsonDocument{{"_id",1},{"title",1},{"score",new BsonDocument("$meta","searchScore")}})}).ToList()

## Hybrid search with RRF

In [ ]:
double Rrf(int rank,int k=60)=>1.0/(k+rank+1);var userQuery="semantic retrieval for rag"; var qv=new BsonArray{0.84,0.76,0.32};var keywordHits=collection.Aggregate<BsonDocument>(new[]{new BsonDocument("$search",new BsonDocument{{"index","idx_body_fts"},{"text",new BsonDocument{{"query",userQuery},{"path","body"}}}}),new BsonDocument("$limit",5),new BsonDocument("$project",new BsonDocument{{"_id",1},{"title",1}})}).ToList();var vectorHits=collection.Aggregate<BsonDocument>(new[]{new BsonDocument("$search",new BsonDocument("cosmosSearch",new BsonDocument{{"path","embedding"},{"vector",qv},{"k",5}})),new BsonDocument("$project",new BsonDocument{{"_id",1},{"title",1}})}).ToList();var scores=new Dictionary<string,double>(); var titles=new Dictionary<string,string>();foreach(var list in new[]{keywordHits,vectorHits}) for(var i=0;i<list.Count;i++){var id=list[i]["_id"].ToString(); titles[id]=list[i]["title"].ToString(); scores[id]=scores.GetValueOrDefault(id)+Rrf(i);}scores.OrderByDescending(x=>x.Value).Take(5).Select(x=>new {Id=x.Key, Title=titles[x.Key], RrfScore=x.Value}).ToList()